In [29]:
import requests
import json
import re
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Optional

def get_uniprot(accession: str) -> requests.Response:
    url = f"https://rest.uniprot.org/uniprotkb/{accession}"
    params = {"format": "json"}
    try:
        response = requests.get(url, params=params)
        return response
    except requests.exceptions.RequestException as e:
        print(f"Error fetching UniProt ID {accession}: {e}")
        return None

def parse_response_uniprot(resp: requests.Response) -> Dict[str, Any]:
    if resp is None or resp.status_code != 200:
        return {}

    data = resp.json()
    try:
        parsed = {
            data['primaryAccession']: {
                "organism": data.get("organism", {}).get("scientificName", "N/A"),
                "geneInfo": data.get("genes", []),
                "sequenceInfo": data.get("sequence", {}),
                "type": data.get("entryType", "protein")
        }}
        return parsed
    except Exception as e:
        print(f"Parsing error for UniProt: {e}")
        return {}

def get_ensembl(id: str) -> requests.Response:
    url = f"https://rest.ensembl.org/lookup/id/{id}"
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.get(url, headers=headers)
        return response
    except requests.exceptions.RequestException as e:
        print(f"Error fetching ENSEMBL ID {id}: {e}")
        return None

def parse_response_ensembl(resp: requests.Response) -> Dict[str, Any]:
    if resp is None or resp.status_code != 200:
        return {}

    data = resp.json()
    try:
        parsed = { 
            data["id"]: {
                "object_type": data.get("object_type"),
                "assembly_name": data.get("assembly_name"),
                "species": data.get("species"),
                "db_type": data.get("db_type"),
                "biotype": data.get("biotype"),
                "display_name": data.get("display_name"),
                "id": data.get("id"),
                "description": data.get("description"),
                "canonical_transcript": data.get("canonical_transcript"),
                "source": data.get("source")
        }}
        return parsed
    except Exception as e:
        print(f"Parsing error for ENSEMBL: {e}")
        return {}

def main(ids: List[str]) -> pd.DataFrame:
    # UniProt - 6 or 10 alphanumeric characters
    # ENSEMBL - starts with ENS followed by object type and 11 digits
    uniprot_re = re.compile(r"^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$")
    ensembl_re = re.compile(r"^ENS[A-Z]*\d{11}$")

    results = []

    for entry_id in ids:
        entry_id = entry_id.strip()
        if not entry_id:
            continue

        print(f"Processing ID: {entry_id}...")

        row = {"query_id": entry_id, "database": "Unknown"}
        parsed_data = {}

        if uniprot_re.match(entry_id):
            row["database"] = "UniProt"
            resp = get_uniprot(entry_id)
            parsed_data = parse_response_uniprot(resp)
        elif ensembl_re.match(entry_id):
            row["database"] = "ENSEMBL"
            resp = get_ensembl(entry_id)
            parsed_data = parse_response_ensembl(resp)
        else:
            print(f"ID {entry_id} did not match known patterns.")

        row.update(parsed_data.get(entry_id, {}))
        results.append(row)

    return pd.DataFrame(results)


In [30]:
get_uniprot('P11473')

<Response [200]>

In [31]:
get_uniprot('helloworld')

<Response [400]>

In [32]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [33]:
parse_response_uniprot(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'UniProtKB reviewed (Swiss-Prot)'}}

In [34]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [35]:
get_ensembl('helloworld')

<Response [400]>

In [36]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [37]:
parse_response_ensembl(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'assembly_name': 'GRCm39',
  'species': 'mus_musculus',
  'db_type': 'core',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'canonical_transcript': 'ENSMUST00000044620.11',
  'source': 'ensembl_havana'}}

In [38]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

Processing ID: P11473...
Processing ID: Q91XI3...
Processing ID: hello...
ID hello did not match known patterns.
Processing ID: ENSG00000157764...
Processing ID: ENSG00000139618...


,query_id,database,organism,geneInfo,sequenceInfo,type,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
0,P11473,UniProt,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q91XI3,UniProt,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hello,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENSG00000157764,ENSEMBL,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
4,ENSG00000139618,ENSEMBL,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
